# Feature-guided Deep Subdomain Adaptation for Steganalysis

Implementation of "Feature-guided Deep Subdomain Adaptation Network for Dataset Mismatch in Spatial Steganalysis"

## Transfer Learning Approach

1. Load pretrained model (trained on BOSSbase MIPOD)
2. Transfer knowledge to target domain (Alaska) via domain adaptation
3. Fine-tune all layers with joint training (classification + LMMD)
4. Evaluate on target domain

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Activation
from tensorflow.keras.layers import GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.layers import AveragePooling2D, MaxPooling2D, Add
from tensorflow.keras.layers import DepthwiseConv2D, Reshape, MultiHeadAttention, LayerNormalization
from tensorflow.keras import regularizers
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## Configuration

In [ ]:
SOURCE_PATH = '/kaggle/input/mipod-npy'
TARGET_PATH = '/kaggle/input/alsk-csnpy/alsk-mipod'
SRM_KERNELS_PATH = '/kaggle/input/stegan/SRM_Kernels.npy'
PRETRAINED_MODEL_PATH = '/kaggle/input/model-keras/SteganBossUni51251202.keras'

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 50
INITIAL_LR = 0.0001
LR_ALPHA = 10
LR_BETA = 0.75

LAMBDA_GAMMA = 10
L2_PENALTY = 1e-5
BN_MOMENTUM = 0.9
DROPOUT_RATE = 0.2

SRM_FEATURE_DIM = 256

SOURCE_DATASET = 'BossMIPOD256x256x50'
TARGET_DATASET = 'alsk-mipod'

print(f"Configuration loaded")
print(f"Source: {SOURCE_PATH}/{SOURCE_DATASET}")
print(f"Target: {TARGET_PATH}/{TARGET_DATASET}")

## Data Loading

In [ ]:
def load_data(base_path, dataset_name, split='train'):
    print(dataset_name)
    if dataset_name in ['alsk-uniw', 'alsk-uerd', 'alsk-mipod']:
        total = 10_000 * 2 # total max data
        splitTTS = (40,10,50) # train, valid, test
        if split=='test':
            st = int(total * splitTTS[2] / 100)
            X = np.load(f'{base_path}/{dataset_name}-X_test.npy')[:st]
            y = np.load(f'{base_path}/{dataset_name}-y_valid.npy')[:st]
            return X, y
        elif split=='valid':
            va = int(total * splitTTS[1] / 100)
            X = np.load(f'{base_path}/{dataset_name}-X_valid.npy')[:va]
            y = np.load(f'{base_path}/{dataset_name}-y_test.npy')[:va]
            return X, y
        else:
            tr = int(total * splitTTS[0] / 100)
            X = np.load(f'{base_path}/{dataset_name}-X_{split}.npy')[:tr]
            y = np.load(f'{base_path}/{dataset_name}-y_{split}.npy')[:tr]
            return X, y
    else:
        X = np.load(f'{base_path}/{dataset_name}-X_{split}.npy')
        y = np.load(f'{base_path}/{dataset_name}-y_{split}.npy')
        return X, y

X_source_train, y_source_train = load_data(SOURCE_PATH, SOURCE_DATASET, 'train')
X_source_valid, y_source_valid = load_data(SOURCE_PATH, SOURCE_DATASET, 'valid')
X_source_test, y_source_test = load_data(SOURCE_PATH, SOURCE_DATASET, 'test')

X_target_train, y_target_train = load_data(TARGET_PATH, TARGET_DATASET, 'train')
X_target_valid, y_target_valid = load_data(TARGET_PATH, TARGET_DATASET, 'valid')
X_target_test, y_target_test = load_data(TARGET_PATH, TARGET_DATASET, 'test')

## Show data
print(f"Source Train: {X_source_train.shape}, {y_source_train.shape}")
print(f"Source Valid: {X_source_valid.shape}, {y_source_valid.shape}")
print(f"Source Test: {X_source_test.shape}, {y_source_test.shape}")
print(f"Target Train: {X_target_train.shape}, {y_target_train.shape}")
print(f"Target Valid: {X_target_valid.shape}, {y_target_test.shape}")
print(f"Target Test: {X_target_test.shape}, {y_target_valid.shape}")

## Tanh3 Activation Function

In [ ]:
def Tanh3(x):
    return tf.tanh(3.0 * x)

## Base Steganalysis Model with Dual Output

Modified version for transfer learning with dual output.

In [ ]:
def build_base_model_dual_output(
    input_shape=(IMG_SIZE, IMG_SIZE, 1),
    srm_kernels_path=SRM_KERNELS_PATH,
    l2_penalty=L2_PENALTY,
    bn_momentum=BN_MOMENTUM,
    dropout_rate=DROPOUT_RATE
):
    srm_weights = np.load(srm_kernels_path)
    biasSRM = np.ones(30, dtype=np.float32)
    
    kernel_reg = regularizers.l2(l2_penalty) if l2_penalty > 0 else None
    
    inputs = Input(shape=input_shape, name="input_image")
    
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    
    fixed_srm_1 = Conv2D(
        30, (5, 5), strides=(1, 1), padding='same',
        activation=Tanh3, use_bias=True,
        kernel_initializer=tf.keras.initializers.Constant(srm_weights),
        bias_initializer=tf.keras.initializers.Constant(biasSRM),
        trainable=False, name="fixed_srm_block_1"
    )(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(fixed_srm_1)
    
    fixed_srm_2 = Conv2D(
        30, (5, 5), strides=(1, 1), padding='same',
        activation=Tanh3, use_bias=True,
        kernel_initializer=tf.keras.initializers.Constant(srm_weights),
        bias_initializer=tf.keras.initializers.Constant(biasSRM),
        trainable=False, name="fixed_srm_block_2"
    )(inputs)
    fixed_srm_2_bn = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(fixed_srm_2)
    
    x = Concatenate(axis=-1, name="srm_combined")([x, fixed_srm_2_bn])
    x = Activation("elu")(x)
    
    res1_skip = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x)
    res1_skip = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_skip)
    res1_skip = Activation("elu")(res1_skip)
    
    res1_main = DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(x)
    res1_main = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res1_main)
    res1_main = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_main)
    res1_main = Activation("elu")(res1_main)
    
    res1_main = DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(res1_main)
    res1_main = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res1_main)
    res1_main = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_main)
    res1_main = Activation("elu")(res1_main)
    
    x = Add()([res1_skip, res1_main])
    
    x = Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    
    x = Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = AveragePooling2D((2, 2))(x)
    
    res2_skip = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x)
    res2_skip = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_skip)
    res2_skip = Activation("elu")(res2_skip)
    
    res2_main = DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(x)
    res2_main = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res2_main)
    res2_main = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_main)
    res2_main = Activation("elu")(res2_main)
    
    res2_main = DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(res2_main)
    res2_main = Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res2_main)
    res2_main = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_main)
    res2_main = Activation("elu")(res2_main)
    
    x = Add()([res2_skip, res2_main])
    
    x = Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = AveragePooling2D((2, 2))(x)
    
    x = Conv2D(90, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = AveragePooling2D((2, 2))(x)
    
    x = Conv2D(120, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = AveragePooling2D((2, 2))(x)
    
    sppf_input = x
    sppf_branches = [sppf_input]
    
    for pool_size_val in [5, 7]:
        pooled = MaxPooling2D(pool_size=pool_size_val, strides=1, padding='same')(sppf_input)
        sppf_branches.append(pooled)
    
    x = Concatenate(axis=-1)(sppf_branches)
    
    x = Conv2D(120, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    
    def mha_block(tensor, num_heads=4):
        h, w, c = tensor.shape[1], tensor.shape[2], tensor.shape[3]
        key_dim = max(1, c // num_heads)
        
        reshaped = Reshape((-1, c))(tensor)
        mha_layer = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        attention_output = mha_layer(reshaped, reshaped)
        mha_reshaped = Reshape((h, w, c))(attention_output)
        out = Add()([tensor, mha_reshaped])
        out = LayerNormalization(epsilon=1e-6)(out)
        return out
    
    x = mha_block(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    
    x = LayerNormalization(epsilon=1e-6)(x)
    
    features = GlobalAveragePooling2D(name="features")(x)
    
    if dropout_rate > 0:
        features_dropout = Dropout(dropout_rate)(features)
    else:
        features_dropout = features
    
    predictions = Dense(2, activation='softmax', kernel_regularizer=kernel_reg, name="predictions")(features_dropout)
    
    model = Model(inputs=inputs, outputs=[predictions, features], name="stegan_dual_output")
    
    return model

base_model = build_base_model_dual_output()
print("Base model with dual output created")

## Load Pretrained Weights (Transfer Learning)

Load weights from model trained on source domain.

In [ ]:
try:
    pretrained_model = keras.models.load_model(
        PRETRAINED_MODEL_PATH,
        custom_objects={'Tanh3': Tanh3}
    )
    
    for layer in base_model.layers:
        if layer.name in [l.name for l in pretrained_model.layers]:
            pretrained_layer = pretrained_model.get_layer(layer.name)
            try:
                layer.set_weights(pretrained_layer.get_weights())
                print(f"Loaded weights for layer: {layer.name}")
            except:
                print(f"Skipped layer (shape mismatch): {layer.name}")
    
    print("Pretrained weights loaded successfully (Transfer Learning)")
except:
    print("No pretrained weights found, training from scratch")

## SRM Feature Extractor

In [ ]:
def build_srm_extractor(input_shape=(IMG_SIZE, IMG_SIZE, 1), output_dim=SRM_FEATURE_DIM):
    srm_weights = np.load(SRM_KERNELS_PATH)
    biasSRM = np.ones(30, dtype=np.float32)
    
    inputs = Input(shape=input_shape)
    
    srm_conv = Conv2D(
        30, (5, 5), strides=(1, 1), padding='same',
        activation=Tanh3, use_bias=True,
        kernel_initializer=tf.keras.initializers.Constant(srm_weights),
        bias_initializer=tf.keras.initializers.Constant(biasSRM),
        trainable=False
    )(inputs)
    
    srm_features = GlobalAveragePooling2D()(srm_conv)
    srm_features_reduced = Dense(output_dim, activation='relu', name="srm_features")(srm_features)
    
    model = Model(inputs=inputs, outputs=srm_features_reduced, name="srm_extractor")
    
    return model

srm_extractor = build_srm_extractor()
print("SRM feature extractor created")

## Feature Fusion Module

In [ ]:
def build_fusion_model(cnn_feature_dim, srm_feature_dim):
    cnn_input = Input(shape=(cnn_feature_dim,), name="cnn_features")
    srm_input = Input(shape=(srm_feature_dim,), name="srm_features")
    
    fused = Concatenate(name="fused_features")([cnn_input, srm_input])
    
    model = Model(inputs=[cnn_input, srm_input], outputs=fused, name="feature_fusion")
    
    return model

cnn_feature_dim = 120
fusion_model = build_fusion_model(cnn_feature_dim, SRM_FEATURE_DIM)
print("Feature fusion module created")

## LMMD Loss Implementation

In [ ]:
def gaussian_kernel(X, Y, gamma=1.0):
    X_norm = tf.reduce_sum(X ** 2, axis=1, keepdims=True)
    Y_norm = tf.reduce_sum(Y ** 2, axis=1, keepdims=True)
    
    distances = X_norm + tf.transpose(Y_norm) - 2.0 * tf.matmul(X, tf.transpose(Y))
    K = tf.exp(-gamma * distances)
    
    return K

def compute_lmmd(source_features, target_features, gamma=1.0):
    n_source = tf.cast(tf.shape(source_features)[0], tf.float32)
    n_target = tf.cast(tf.shape(target_features)[0], tf.float32)
    
    K_ss = gaussian_kernel(source_features, source_features, gamma)
    K_tt = gaussian_kernel(target_features, target_features, gamma)
    K_st = gaussian_kernel(source_features, target_features, gamma)
    
    term1 = tf.reduce_sum(K_ss) / (n_source ** 2)
    term2 = tf.reduce_sum(K_tt) / (n_target ** 2)
    term3 = tf.reduce_sum(K_st) / (n_source * n_target)
    
    lmmd = term1 + term2 - 2.0 * term3
    
    return lmmd

def compute_subdomain_lmmd(
    source_features, source_labels,
    target_features, target_predictions,
    gamma=1.0
):
    source_labels_idx = tf.argmax(source_labels, axis=1)
    target_labels_idx = tf.argmax(target_predictions, axis=1)
    
    source_cover_mask = tf.equal(source_labels_idx, 0)
    source_stego_mask = tf.equal(source_labels_idx, 1)
    target_cover_mask = tf.equal(target_labels_idx, 0)
    target_stego_mask = tf.equal(target_labels_idx, 1)
    
    source_cover = tf.boolean_mask(source_features, source_cover_mask)
    source_stego = tf.boolean_mask(source_features, source_stego_mask)
    target_cover = tf.boolean_mask(target_features, target_cover_mask)
    target_stego = tf.boolean_mask(target_features, target_stego_mask)
    
    lmmd_cover = tf.cond(
        tf.logical_and(tf.shape(source_cover)[0] > 0, tf.shape(target_cover)[0] > 0),
        lambda: compute_lmmd(source_cover, target_cover, gamma),
        lambda: 0.0
    )
    
    lmmd_stego = tf.cond(
        tf.logical_and(tf.shape(source_stego)[0] > 0, tf.shape(target_stego)[0] > 0),
        lambda: compute_lmmd(source_stego, target_stego, gamma),
        lambda: 0.0
    )
    
    total_lmmd = (lmmd_cover + lmmd_stego) / 2.0
    
    return total_lmmd

print("LMMD loss function defined")

## Corrected Adaptive Lambda Scheduler

Based on DSAN paper reference: lambda = 2/(1 + exp(-gamma*theta)) - 1

In [ ]:
def adaptive_lambda(epoch, total_epochs, gamma=LAMBDA_GAMMA):
    theta = epoch / total_epochs
    lambda_val = 2.0 / (1.0 + np.exp(-gamma * theta)) - 1.0
    return lambda_val

epochs_test = np.arange(0, EPOCHS)
lambdas_test = [adaptive_lambda(e, EPOCHS) for e in epochs_test]

plt.figure(figsize=(10, 4))
plt.plot(epochs_test, lambdas_test)
plt.xlabel('Epoch')
plt.ylabel('Lambda')
plt.title('Adaptive Lambda Schedule (Corrected)')
plt.grid(True)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.3)
plt.axhline(y=1, color='r', linestyle='--', alpha=0.3)
plt.show()

print(f"Lambda at epoch 0: {adaptive_lambda(0, EPOCHS):.4f}")
print(f"Lambda at epoch 50: {adaptive_lambda(50, EPOCHS):.4f}")
print(f"Lambda at epoch 100: {adaptive_lambda(100, EPOCHS):.4f}")
print(f"Lambda at epoch 200: {adaptive_lambda(200, EPOCHS):.4f}")

## Learning Rate Schedule

Paper formula: lr = lr_0 / (1 + alpha*theta)^beta

In [ ]:
def lr_schedule(epoch, total_epochs, lr_0=INITIAL_LR, alpha=LR_ALPHA, beta=LR_BETA):
    theta = epoch / total_epochs
    lr = lr_0 / ((1.0 + alpha * theta) ** beta)
    return lr

epochs_test = np.arange(0, EPOCHS)
lrs_test = [lr_schedule(e, EPOCHS) for e in epochs_test]

plt.figure(figsize=(10, 4))
plt.plot(epochs_test, lrs_test)
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.grid(True)
plt.yscale('log')
plt.show()

print(f"LR at epoch 0: {lr_schedule(0, EPOCHS):.6f}")
print(f"LR at epoch 100: {lr_schedule(100, EPOCHS):.6f}")
print(f"LR at epoch 200: {lr_schedule(200, EPOCHS):.6f}")

## Domain Adaptation Training Loop with Transfer Learning

In [ ]:
class DomainAdaptationTrainer:
    def __init__(
        self,
        base_model,
        srm_extractor,
        fusion_model,
        initial_lr=INITIAL_LR
    ):
        self.base_model = base_model
        self.srm_extractor = srm_extractor
        self.fusion_model = fusion_model
        
        self.optimizer = tf.keras.optimizers.AdamW(learning_rate=initial_lr)
        
        self.loss_cls_metric = tf.keras.metrics.Mean(name='loss_cls')
        self.loss_lmmd_metric = tf.keras.metrics.Mean(name='loss_lmmd')
        self.loss_total_metric = tf.keras.metrics.Mean(name='loss_total')
        self.source_accuracy_metric = tf.keras.metrics.CategoricalAccuracy(name='source_acc')
        
        self.history = {
            'loss_cls': [],
            'loss_lmmd': [],
            'loss_total': [],
            'source_acc': [],
            'target_acc': [],
            'lambda': [],
            'lr': []
        }
    
    @tf.function
    def train_step(self, source_batch, target_batch, lambda_val):
        X_source, y_source = source_batch
        X_target = target_batch
        
        with tf.GradientTape() as tape:
            pred_source, feat_source = self.base_model(X_source, training=True)
            pred_target, feat_target = self.base_model(X_target, training=True)
            
            srm_source = self.srm_extractor(X_source, training=True)
            srm_target = self.srm_extractor(X_target, training=True)
            
            fused_source = self.fusion_model([feat_source, srm_source], training=True)
            fused_target = self.fusion_model([feat_target, srm_target], training=True)
            
            loss_cls = tf.keras.losses.categorical_crossentropy(y_source, pred_source)
            loss_cls = tf.reduce_mean(loss_cls)
            
            loss_lmmd = compute_subdomain_lmmd(
                fused_source, y_source,
                fused_target, pred_target,
                gamma=1.0
            )
            
            lambda_val_tf = tf.cast(lambda_val, tf.float32)
            loss_total = loss_cls + lambda_val_tf * loss_lmmd
        
        trainable_vars = (
            self.base_model.trainable_variables +
            self.srm_extractor.trainable_variables +
            self.fusion_model.trainable_variables
        )
        
        gradients = tape.gradient(loss_total, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        
        self.loss_cls_metric.update_state(loss_cls)
        self.loss_lmmd_metric.update_state(loss_lmmd)
        self.loss_total_metric.update_state(loss_total)
        self.source_accuracy_metric.update_state(y_source, pred_source)
        
        return loss_cls, loss_lmmd, loss_total
    
    def evaluate_target(self, X_target, y_target, batch_size=32):
        n_samples = len(X_target)
        all_pred_labels = []
        all_true_labels = []
        
        for i in range(0, n_samples, batch_size):
            batch_X = X_target[i:i+batch_size]
            batch_y = y_target[i:i+batch_size]
            
            pred_target, _ = self.base_model(batch_X, training=False)
            
            pred_labels = tf.argmax(pred_target, axis=1)
            true_labels = tf.argmax(batch_y, axis=1)
            
            all_pred_labels.extend(pred_labels.numpy())
            all_true_labels.extend(true_labels.numpy())
        
        all_pred_labels = np.array(all_pred_labels)
        all_true_labels = np.array(all_true_labels)
        
        cover_mask = all_true_labels == 0
        stego_mask = all_true_labels == 1
        
        cover_correct = np.sum((cover_mask) & (all_pred_labels == 0))
        cover_total = np.sum(cover_mask)
        cover_acc = cover_correct / cover_total if cover_total > 0 else 0.0
        
        stego_correct = np.sum((stego_mask) & (all_pred_labels == 1))
        stego_total = np.sum(stego_mask)
        stego_acc = stego_correct / stego_total if stego_total > 0 else 0.0
        
        P_A = (cover_acc + stego_acc) / 2.0
        
        return P_A
    
    def train(
        self,
        X_source_train, y_source_train,
        X_target_train,
        X_target_valid, y_target_valid,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    ):
        n_source = len(X_source_train)
        n_target = len(X_target_train)
        n_batches = min(n_source, n_target) // batch_size
        
        for epoch in range(epochs):
            indices_source = np.random.permutation(n_source)
            indices_target = np.random.permutation(n_target)
            
            self.loss_cls_metric.reset_state()
            self.loss_lmmd_metric.reset_state()
            self.loss_total_metric.reset_state()
            self.source_accuracy_metric.reset_state()
            
            lambda_val = adaptive_lambda(epoch, epochs)
            current_lr = lr_schedule(epoch, epochs)
            self.optimizer.learning_rate.assign(current_lr)
            
            for batch_idx in range(n_batches):
                start_idx = batch_idx * batch_size
                end_idx = start_idx + batch_size
                
                batch_indices_source = indices_source[start_idx:end_idx]
                batch_indices_target = indices_target[start_idx:end_idx]
                
                X_source_batch = X_source_train[batch_indices_source]
                y_source_batch = y_source_train[batch_indices_source]
                X_target_batch = X_target_train[batch_indices_target]
                
                source_batch = (X_source_batch, y_source_batch)
                target_batch = X_target_batch
                
                self.train_step(source_batch, target_batch, lambda_val)
            
            loss_cls = self.loss_cls_metric.result().numpy()
            loss_lmmd = self.loss_lmmd_metric.result().numpy()
            loss_total = self.loss_total_metric.result().numpy()
            source_acc = self.source_accuracy_metric.result().numpy()
            
            target_acc = self.evaluate_target(X_target_valid, y_target_valid, batch_size=batch_size)
            
            self.history['loss_cls'].append(loss_cls)
            self.history['loss_lmmd'].append(loss_lmmd)
            self.history['loss_total'].append(loss_total)
            self.history['source_acc'].append(source_acc)
            self.history['target_acc'].append(target_acc)
            self.history['lambda'].append(lambda_val)
            self.history['lr'].append(current_lr)
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}/{epochs}")
                print(f"  Loss Cls: {loss_cls:.4f} | Loss LMMD: {loss_lmmd:.4f} | Loss Total: {loss_total:.4f}")
                print(f"  Source Acc: {source_acc:.4f} | Target Acc: {target_acc:.4f}")
                print(f"  Lambda: {lambda_val:.4f} | LR: {current_lr:.6f}")
        
        return self.history

print("Domain Adaptation Trainer with Transfer Learning defined")

## Initialize and Train

In [ ]:
trainer = DomainAdaptationTrainer(
    base_model=base_model,
    srm_extractor=srm_extractor,
    fusion_model=fusion_model,
    initial_lr=INITIAL_LR
)

print("Starting domain adaptation training with transfer learning...")

history = trainer.train(
    X_source_train, y_source_train,
    X_target_train,
    X_target_valid, y_target_valid,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

print("Training completed")

## Evaluation on Target Test Set

In [ ]:
target_test_acc = trainer.evaluate_target(X_target_test, y_target_test, batch_size=BATCH_SIZE)

print(f"Final Target Test Accuracy (P_A): {target_test_acc:.4f}")

all_pred_labels = []
all_true_labels = []

for i in range(0, len(X_target_test), BATCH_SIZE):
    batch_X = X_target_test[i:i+BATCH_SIZE]
    batch_y = y_target_test[i:i+BATCH_SIZE]
    
    pred_target_batch, _ = base_model(batch_X, training=False)
    
    pred_labels = np.argmax(pred_target_batch.numpy(), axis=1)
    true_labels = np.argmax(batch_y, axis=1)
    
    all_pred_labels.extend(pred_labels)
    all_true_labels.extend(true_labels)

all_pred_labels = np.array(all_pred_labels)
all_true_labels = np.array(all_true_labels)

cm = confusion_matrix(all_true_labels, all_pred_labels)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(all_true_labels, all_pred_labels, target_names=['Cover', 'Stego'], digits=4))

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(history['loss_cls'], label='Classification Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Classification Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['loss_lmmd'], label='LMMD Loss', color='orange')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('LMMD Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[0, 2].plot(history['loss_total'], label='Total Loss', color='red')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].set_title('Total Loss')
axes[0, 2].legend()
axes[0, 2].grid(True)

axes[1, 0].plot(history['source_acc'], label='Source Accuracy')
axes[1, 0].plot(history['target_acc'], label='Target Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history['lambda'], label='Lambda', color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Lambda Value')
axes[1, 1].set_title('Adaptive Lambda Schedule')
axes[1, 1].legend()
axes[1, 1].grid(True)

axes[1, 2].plot(history['lr'], label='Learning Rate', color='purple')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Learning Rate')
axes[1, 2].set_title('Learning Rate Schedule')
axes[1, 2].set_yscale('log')
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

## Save Adapted Model

In [ ]:
base_model.save('domain_adapted_stegan_model.keras')
srm_extractor.save('srm_extractor.keras')
fusion_model.save('fusion_model.keras')

np.save('training_history.npy', history)

print("Models and training history saved successfully")